# HighRes-Net Training on Microscopy Data

This notebook trains HighRes-Net on your TIFF microscopy dataset (7 LR frames → 2x HR upsampling).

**Format:**
- ✓ TIFF 8-bit grayscale files (LR_1.tif/tiff through LR_7.tif/tiff + HR.tif/tiff per image)
- ✓ Pre-augmented images in D:\GUC\Datasets\HighRes input test\image_0000\, image_0001\, etc.
- ✓ 2x upsampling (128×128 LR → 256×256 HR)
- ✓ Supports both .tif and .tiff extensions (identical format)
- ✓ GPU acceleration with RTX 4060

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import time

sys.path.insert(0, '../src')

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from DataLoader import TiffPatchDataset, collateFunction
from DeepNetworks.HRNet import HRNet

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Verify Dataset

In [ ]:
data_root = Path("D:\\GUC\\Datasets\\HighRes Train input")
# Discover image folders by contents (contain LR_*.tif* and HR.tif*) rather than by name prefix
image_dirs = sorted([d for d in data_root.iterdir() if d.is_dir() and any(d.glob('LR_*.tif*')) and any(d.glob('HR.tif*'))])

print(f"✓ Found {len(image_dirs)} images")

# Verify first image has required files
if image_dirs:
    first = image_dirs[0]
    lr_files = sorted(first.glob("LR_*.tif*"))
    hr_file = list(first.glob("HR.tif*"))
    print(f"✓ Sample ({first.name}): {len(lr_files)} LR frames, {len(hr_file)} HR frame")

## 3. Load Configuration

In [ ]:
config_path = Path('../config/config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

print("Training Configuration:")
print(f"  Num epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['lr']}")
print(f"  Patch size: {config['training']['patch_size']}")
print(f"  N_views: {config['training']['n_views']}")
print(f"  Create patches: {config['training']['create_patches']}")

## 5. Create Datasets and DataLoaders

In [ ]:
print("Creating TIFF dataset...")

# Discover all image directories by checking for expected LR/HR files
train_root = Path("D:\\GUC\\Datasets\\HighRes Train input")
val_root = Path("D:\\GUC\\Datasets\\lights dataset\\validation sets")

train_image_dirs = sorted([
    str(d) for d in train_root.iterdir()
    if d.is_dir() and any(d.glob('LR_*.tif*')) and any(d.glob('HR.tif*'))
 ])
val_image_dirs = sorted([
    str(d) for d in val_root.iterdir()
    if d.is_dir() and any(d.glob('LR_*.tif*')) and any(d.glob('HR.tif*'))
 ]) if val_root.exists() else []

print(f"Discovered train images: {len(train_image_dirs)}")
print(f"Discovered val images: {len(val_image_dirs)}")

# Create dataset from images
train_dataset = TiffPatchDataset(
    patch_dirs=train_image_dirs,
    max_views=config['training']['n_views']
 )
val_dataset = TiffPatchDataset(
    patch_dirs=val_image_dirs,
    max_views=config['training']['n_views']
 ) if val_image_dirs else None

print(f"Training dataset size: {len(train_dataset)} images")
print(f"Validation dataset size: {len(val_dataset) if val_dataset is not None else 0} images")

# Create DataLoader
batch_size = config['training']['batch_size']
min_L = config['training']['min_L']
n_workers = int(config['training'].get('n_workers', 0))
pin_memory = torch.cuda.is_available()
use_workers = n_workers > 0
loader_kwargs = {
    'num_workers': n_workers,
    'pin_memory': pin_memory,
}
if use_workers:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2,
    })

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collateFunction(min_L=min_L),
    **loader_kwargs,
 )

if val_dataset is not None:
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collateFunction(min_L=min_L),
        **loader_kwargs,
    )
else:
    val_loader = None

print(f"DataLoader created:")
print(f"  batch_size: {batch_size}")
print(f"  train_batches: {len(train_loader)}")
print(f"  val_batches: {len(val_loader) if val_loader is not None else 0}")
print(f"  num_workers: {n_workers}")
print(f"  pin_memory: {pin_memory}")
if use_workers:
    print("  persistent_workers: True  prefetch_factor: 2")

## 6. Initialize Model

In [ ]:
print("Initializing model...")

model = HRNet(config['network'])
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Model device: {device}")

## 7. Setup Optimizer and Loss

In [ ]:
print("Setting up training...")
def charbonnier_loss(sr, hr, eps=1e-3):
    return torch.mean(torch.sqrt((sr - hr) ** 2 + eps ** 2))
criterion = charbonnier_loss

from DeepNetworks.cbam import CBAM
from DeepNetworks.HRNet import FusionAttention

network_cfg        = config.get("network", {})
USE_CBAM           = bool(network_cfg.get("use_cbam", False))
USE_CBAM_WARMUP    = bool(config['training'].get('use_cbam_warmup', False)) and USE_CBAM
CBAM_WARMUP_EPOCHS = int(config['training'].get('cbam_warmup_epochs', 10))
BASE_LR_JOINT      = float(config['training'].get('base_lr_joint', 3e-6))
CBAM_LR_WARMUP     = float(config['training'].get('cbam_lr_warmup', 3e-4))
CBAM_LR_JOINT      = float(config['training'].get('cbam_lr_joint', 5e-5))
FUSION_LR          = float(config['training'].get('fusion_lr', 3e-4))
FUSION_BLEND_LR    = float(config['training'].get('fusion_blend_lr', 3e-3))

USE_AMP            = bool(config['training'].get('use_amp', True)) and torch.cuda.is_available()
AMP_DTYPE_NAME     = str(config['training'].get('amp_dtype', 'float16')).lower()
GRAD_ACCUM_STEPS   = max(1, int(config['training'].get('grad_accum_steps', 1)))

if AMP_DTYPE_NAME in ('bf16', 'bfloat16'):
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

# ---------------------------------------------------------------------------
# Four-group parameter split: base / CBAM / fusion core / fusion blend gate
# Blend gate gets higher LR to exit passthrough regime faster.
# ---------------------------------------------------------------------------
def _split_four_param_groups(model_instance):
    cbam_ids = set()
    fusion_core_ids = set()
    fusion_blend_ids = set()

    for module in model_instance.modules():
        if isinstance(module, CBAM):
            for p in module.parameters():
                cbam_ids.add(id(p))
        elif isinstance(module, FusionAttention):
            fusion_blend_ids.add(id(module.blend))
            for p in module.parameters():
                if id(p) != id(module.blend):
                    fusion_core_ids.add(id(p))

    base_params, cbam_params, fusion_core_params, fusion_blend_params = [], [], [], []
    for p in model_instance.parameters():
        pid = id(p)
        if pid in fusion_blend_ids:
            fusion_blend_params.append(p)
        elif pid in fusion_core_ids:
            fusion_core_params.append(p)
        elif pid in cbam_ids:
            cbam_params.append(p)
        else:
            base_params.append(p)

    return base_params, cbam_params, fusion_core_params, fusion_blend_params

base_params, cbam_params, fusion_core_params, fusion_blend_params = _split_four_param_groups(model)
cbam_warmup_done = False

print("Parameter groups:")
print(f"  Base params:         {sum(p.numel() for p in base_params):,}  lr={BASE_LR_JOINT}")
print(f"  CBAM params:         {sum(p.numel() for p in cbam_params):,}  lr={CBAM_LR_JOINT}")
print(f"  Fusion core params:  {sum(p.numel() for p in fusion_core_params):,}  lr={FUSION_LR}")
print(f"  Fusion blend params: {sum(p.numel() for p in fusion_blend_params):,}  lr={FUSION_BLEND_LR}")
print(f"  use_cbam: {USE_CBAM}  use_cbam_warmup: {USE_CBAM_WARMUP}")
print(f"  amp: {USE_AMP}  amp_dtype: {AMP_DTYPE_NAME}  grad_accum_steps: {GRAD_ACCUM_STEPS}")

if len(fusion_core_params) == 0:
    print("  WARNING: FusionAttention core params not found — check HRNet.py and restart kernel")
if len(fusion_blend_params) == 0:
    print("  WARNING: FusionAttention blend param not found — check HRNet.py and restart kernel")

for p in model.parameters():
    p.requires_grad = True

optimizer = optim.Adam([
    {'params': base_params,         'lr': BASE_LR_JOINT},
    {'params': cbam_params,         'lr': CBAM_LR_JOINT},
    {'params': fusion_core_params,  'lr': FUSION_LR},
    {'params': fusion_blend_params, 'lr': FUSION_BLEND_LR},
], betas=(0.9, 0.999))

print("Optimizer: four-group joint (base/CBAM/fusion-core/fusion-blend)")

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['training']['lr_step'],
    gamma=config['training']['lr_decay']
)
print(f"LR decay: {config['training']['lr_decay']} every {config['training']['lr_step']} epochs")
print("Loss function: Charbonnier Loss")

## 8. Training Loop

In [ ]:
# Training run setup (selection is managed by notebooks/weights_manager.ipynb)
import json
from datetime import datetime
from pathlib import Path



WEIGHTS_ROOT = Path("../models/weights")
RUNS_TMP_DIR = WEIGHTS_ROOT / "_runs_tmp"
ACTIVE_SELECTION_PATH = WEIGHTS_ROOT / "active_weight_selection.json"
LAST_TRAINING_CANDIDATE_PATH = WEIGHTS_ROOT / "last_training_candidate.json"

for folder in [WEIGHTS_ROOT, RUNS_TMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def _read_active_selection():
    if not ACTIVE_SELECTION_PATH.exists():
        return None
    try:
        with open(ACTIVE_SELECTION_PATH, "r") as f:
            data = json.load(f)
        path_value = data.get("path")
        if not path_value:
            return None
        candidate = Path(path_value)
        if candidate.exists():
            return candidate
    except Exception:
        return None
    return None


# ---------------------------------------------------------------------------
# USER CONTROLS (edit these only)
# ---------------------------------------------------------------------------
RUN_TAG = ""
ALLOW_SCRATCH_IF_NO_SELECTION = True  # If False, training stops unless weights_manager set an active weight

# Architecture family is inferred from config switch for consistency.
network_cfg = config.get("network", {})
USE_CBAM = bool(network_cfg.get("use_cbam", False))
ACTIVE_ARCH_FAMILY = "CBAM" if USE_CBAM else "Base"

selected_source_weight_path = _read_active_selection()
selected_source_weight_name = selected_source_weight_path.stem if selected_source_weight_path else None

if selected_source_weight_path is not None:
    print(f"Selected source weight from weights manager: {selected_source_weight_path}")
else:
    if not ALLOW_SCRATCH_IF_NO_SELECTION:
        raise RuntimeError(
            "No active weight selection found. Set one in notebooks/weights_manager.ipynb or enable scratch mode."
        )
    print("No active weight selected -> training from scratch")
    

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_init = "finetune" if selected_source_weight_path else "scratch"
run_name = f"{run_stamp}_{ACTIVE_ARCH_FAMILY.lower()}_{run_init}_{RUN_TAG}"
run_dir = RUNS_TMP_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

managed_best_weights_path = run_dir / "best.pth"
managed_checkpoint_dir = run_dir / "checkpoints"
managed_checkpoint_dir.mkdir(parents=True, exist_ok=True)
managed_checkpoint_meta_path = run_dir / "training_metadata.json"

print("\nRun setup:")
print(f"  run_name: {run_name}")
print(f"  run_dir: {run_dir}")
print(f"  architecture: {ACTIVE_ARCH_FAMILY} (use_cbam={USE_CBAM})")
print(f"  mode: {run_init}")
print("  save/discard: handled in notebooks/weights_manager.ipynb")


In [ ]:
"""
HighRes-Net Training Cell (Cell 8) - VRAM-optimized + FusionAttention-aware version.
Requires: _split_four_param_groups defined in Cell 7.
"""

from tqdm import tqdm
import json as json_module
import sys
from datetime import datetime
from contextlib import nullcontext

# ============================================================================
# Bridge variables from run-setup cell
# ============================================================================
best_weights_path    = managed_best_weights_path
checkpoint_meta_path = managed_checkpoint_meta_path
checkpoint_dir       = managed_checkpoint_dir

# ============================================================================
# AMP / accumulation setup (VRAM optimization)
# ============================================================================
scaler_enabled = bool(USE_AMP and AMP_DTYPE == torch.float16)
scaler = torch.amp.GradScaler('cuda', enabled=scaler_enabled)

# ============================================================================
# Loss functions
# ============================================================================

def compute_range_loss(sr_output, hr_target):
    sr_min, sr_max = sr_output.min(), sr_output.max()
    hr_min, hr_max = hr_target.min(), hr_target.max()
    range_loss = ((sr_max - sr_min) - 0.6) ** 2
    min_align  = (sr_min - hr_min) ** 2
    max_align  = (sr_max - hr_max) ** 2
    return range_loss * 0.5 + (min_align + max_align) * 0.25

def tv_loss(output):
    diff_h = torch.abs(output[:, :, 1:,  :] - output[:, :, :-1, :])
    diff_w = torch.abs(output[:, :,  :, 1:] - output[:, :,  :, :-1])
    return torch.mean(diff_h) + torch.mean(diff_w)

def edge_aware_tv_loss(output, hr_target, edge_k=10.0):
    hr_grad_h = torch.abs(hr_target[:, :, 1:,  :] - hr_target[:, :, :-1, :])
    hr_grad_w = torch.abs(hr_target[:, :,  :, 1:] - hr_target[:, :,  :, :-1])
    edge_weight_h = torch.exp(-edge_k * hr_grad_h)
    edge_weight_w = torch.exp(-edge_k * hr_grad_w)
    sr_grad_h = torch.abs(output[:, :, 1:,  :] - output[:, :, :-1, :])
    sr_grad_w = torch.abs(output[:, :,  :, 1:] - output[:, :,  :, :-1])
    return torch.mean(edge_weight_h * sr_grad_h) + torch.mean(edge_weight_w * sr_grad_w)

def _radial_frequency_mask(h, w, low_ratio, high_ratio, device, dtype):
    fy = torch.fft.fftfreq(h, d=1.0, device=device).view(h, 1)
    fx = torch.fft.fftfreq(w, d=1.0, device=device).view(1, w)
    r  = torch.sqrt(fy * fy + fx * fx)
    r  = r / r.max().clamp_min(1e-8)
    return ((r >= low_ratio) & (r <= high_ratio)).to(dtype)

def freq_band_loss(sr_output, hr_target, low_ratio=0.08, high_ratio=0.35, use_log_magnitude=True):
    err     = sr_output - hr_target
    err_fft = torch.fft.fft2(err, dim=(-2, -1))
    err_mag = torch.abs(err_fft)
    if use_log_magnitude:
        err_mag = torch.log1p(err_mag)
    h, w  = err_mag.shape[-2], err_mag.shape[-1]
    band  = _radial_frequency_mask(h, w, low_ratio, high_ratio, err_mag.device, err_mag.dtype)
    band  = band.unsqueeze(0).unsqueeze(0)
    denom = band.sum() * err_mag.shape[0] * err_mag.shape[1]
    return ((err_mag * band) ** 2).sum() / denom.clamp_min(1.0)

def directional_grad_loss(sr_output, hr_target, emphasis_axis='none', emphasis=1.2):
    sr_dx = sr_output[:, :, :, 1:] - sr_output[:, :, :, :-1]
    sr_dy = sr_output[:, :, 1:, :] - sr_output[:, :, :-1, :]
    hr_dx = hr_target[:, :, :, 1:] - hr_target[:, :, :, :-1]
    hr_dy = hr_target[:, :, 1:, :] - hr_target[:, :, :-1, :]

    loss_x = torch.mean(torch.abs(sr_dx - hr_dx))
    loss_y = torch.mean(torch.abs(sr_dy - hr_dy))

    if emphasis_axis == 'x':
        total = emphasis * loss_x + loss_y
    elif emphasis_axis == 'y':
        total = loss_x + emphasis * loss_y
    else:
        total = loss_x + loss_y

    return total, loss_x, loss_y

def _axis_frequency_weight_mask(h, w, axis_sigma_ratio, device, dtype):
    fy = torch.fft.fftfreq(h, d=1.0, device=device).view(h, 1)
    fx = torch.fft.fftfreq(w, d=1.0, device=device).view(1, w)

    fy = fy / fy.abs().max().clamp_min(1e-8)
    fx = fx / fx.abs().max().clamp_min(1e-8)

    sigma = max(1e-4, float(axis_sigma_ratio))
    near_x_axis = torch.exp(-(fy * fy) / (2.0 * sigma * sigma))
    near_y_axis = torch.exp(-(fx * fx) / (2.0 * sigma * sigma))

    return torch.maximum(near_x_axis, near_y_axis).to(dtype)

def axis_freq_loss(sr_output, hr_target, low_ratio=0.08, high_ratio=0.35, axis_sigma_ratio=0.08, use_log_magnitude=True):
    err     = sr_output - hr_target
    err_fft = torch.fft.fft2(err, dim=(-2, -1))
    err_mag = torch.abs(err_fft)

    if use_log_magnitude:
        err_mag = torch.log1p(err_mag)

    h, w = err_mag.shape[-2], err_mag.shape[-1]
    band = _radial_frequency_mask(h, w, low_ratio, high_ratio, err_mag.device, err_mag.dtype)
    axis_weight = _axis_frequency_weight_mask(h, w, axis_sigma_ratio, err_mag.device, err_mag.dtype)

    weight = (band * axis_weight).unsqueeze(0).unsqueeze(0)
    denom = weight.sum() * err_mag.shape[0] * err_mag.shape[1]

    return ((err_mag * weight) ** 2).sum() / denom.clamp_min(1.0)

# ============================================================================
# Diagnostics - CBAM gates + FusionAttention blend gate
# ============================================================================

cbam_diag   = {"channel_means": [], "channel_stds": [], "spatial_means": [], "spatial_stds": []}
fusion_diag = {"blend_values": []}
all_hooks   = []

def _cbam_gate_hook(kind):
    def _hook(module, _inputs, output):
        out = output.detach()
        cbam_diag[f"{kind}_means"].append(float(out.mean()))
        cbam_diag[f"{kind}_stds"].append(float(out.std()))
    return _hook

def _fusion_blend_hook(module, _inputs, _output):
    fusion_diag["blend_values"].append(float(torch.sigmoid(module.blend).item()))

if USE_CBAM:
    for name, module in model.named_modules():
        if name.endswith("channel_attn.gate"):
            all_hooks.append(module.register_forward_hook(_cbam_gate_hook("channel")))
        elif name.endswith("spatial_attn.gate"):
            all_hooks.append(module.register_forward_hook(_cbam_gate_hook("spatial")))
    print(f"CBAM diagnostics: {len(all_hooks)} gate hooks registered")

fusion_hook_registered = False
for name, module in model.named_modules():
    if hasattr(module, 'blend') and hasattr(module, 'query'):
        all_hooks.append(module.register_forward_hook(_fusion_blend_hook))
        print(f"FusionAttention blend hook registered on: {name}")
        fusion_hook_registered = True
        break

if not fusion_hook_registered:
    print("WARNING: FusionAttention not found - check HRNet.py and restart kernel.")

# ============================================================================
# Training config
# ============================================================================

num_epochs      = config['training']['num_epochs']
lambda_range    = config['training'].get('lambda_range', 0.003)
smoothness_mode = config['training'].get('smoothness_mode', 'tv').lower()
lambda_tv       = config['training'].get('lambda_tv', 0.0)
lambda_edge_tv  = config['training'].get('lambda_edge_tv', 0.0)
lambda_freq     = config['training'].get('lambda_freq', 0.0)
edge_aware_k    = float(config['training'].get('edge_aware_k', 10.0))
freq_low_ratio  = float(config['training'].get('freq_low_ratio', 0.08))
freq_high_ratio = float(config['training'].get('freq_high_ratio', 0.35))
freq_use_log    = bool(config['training'].get('freq_use_log_magnitude', True))

# Direction-aware anti-alias controls
lambda_dir         = float(config['training'].get('lambda_dir', 0.002))
dir_emphasis_axis  = str(config['training'].get('dir_emphasis_axis', 'none')).lower()
dir_emphasis       = float(config['training'].get('dir_emphasis', 1.2))
lambda_axis_freq   = float(config['training'].get('lambda_axis_freq', 0.003))
axis_sigma_ratio   = float(config['training'].get('axis_sigma_ratio', 0.08))

# If loading a legacy checkpoint without fusion attention keys,
# nudge blend out of near-zero passthrough to allow attention learning.
FUSION_BLEND_INIT_IF_LEGACY = float(config['training'].get('fusion_blend_init_if_legacy', -4.5))

lambda_smooth = {
    'edge_aware_tv': lambda_edge_tv if lambda_edge_tv > 0 else lambda_tv,
    'tv':            lambda_tv,
    'freq_band':     lambda_freq,
}.get(smoothness_mode, 0.0)

print("\nTraining config:")
print(f"  smoothness: {smoothness_mode}  lambda: {lambda_smooth}")
print(f"  freq_band: [{freq_low_ratio}, {freq_high_ratio}]  log: {freq_use_log}")
print(f"  directional grad: lambda={lambda_dir}  axis={dir_emphasis_axis}  emphasis={dir_emphasis}")
print(f"  axis-freq: lambda={lambda_axis_freq}  sigma={axis_sigma_ratio}")
print(f"  lambda_range: {lambda_range}  epochs: {num_epochs}")
print(f"  amp: {USE_AMP} ({AMP_DTYPE_NAME})  scaler: {scaler_enabled}  accum: {GRAD_ACCUM_STEPS}")
print(f"  best_weights -> {best_weights_path}")

# ============================================================================
# Source weight load (scratch-safe)
# ============================================================================
source_has_fusion_keys = False
if selected_source_weight_path is not None:
    source_raw = torch.load(selected_source_weight_path, map_location=device, weights_only=False)

    if isinstance(source_raw, dict) and 'state_dict' in source_raw:
        source_sd = source_raw['state_dict']
    elif isinstance(source_raw, dict) and 'model_state_dict' in source_raw:
        source_sd = source_raw['model_state_dict']
    else:
        source_sd = source_raw

    if not isinstance(source_sd, dict):
        raise RuntimeError('Source checkpoint is not a valid state_dict dictionary')

    source_sd = {
        (k[7:] if isinstance(k, str) and k.startswith('module.') else k): v
        for k, v in source_sd.items()
    }

    source_has_fusion_keys = any(k.startswith('fuse.fusion_attn.') for k in source_sd.keys())

    model_sd  = model.state_dict()
    filtered, skipped = {}, []
    for key, tensor in source_sd.items():
        if key not in model_sd:
            skipped.append(f"{key}: not in model")
        elif tensor.shape != model_sd[key].shape:
            skipped.append(f"{key}: shape {tuple(tensor.shape)} != {tuple(model_sd[key].shape)}")
        else:
            filtered[key] = tensor
    model.load_state_dict(filtered, strict=False)



    # Legacy-fusion kickstart: checkpoint has no fusion attention weights.
    if not source_has_fusion_keys:
        for module in model.modules():
            if isinstance(module, FusionAttention):
                with torch.no_grad():
                    module.blend.fill_(FUSION_BLEND_INIT_IF_LEGACY)
                print(f"  Legacy checkpoint detected: set fusion blend init to {FUSION_BLEND_INIT_IF_LEGACY}")
                break

    enc_std = model.encode.init_layer[0].weight.std().item()
    print(f"\nSource load: {selected_source_weight_path.name}")
    print(f"  Keys loaded: {len(filtered)}  skipped: {len(skipped)}")
    print(f"  source_has_fusion_keys: {source_has_fusion_keys}")
    for s in skipped[:5]:
        print(f"    {s}")
    print(f"  Encoder std: {enc_std:.6f}  {'trained' if enc_std > 0.02 else 'may be random'}")
else:
    print("\nNo source weight - training from scratch.")

# ============================================================================
# Rebuild optimizer AFTER weight load using four-group split
# ============================================================================

base_params, cbam_params, fusion_core_params, fusion_blend_params = _split_four_param_groups(model)

for p in model.parameters():
    p.requires_grad = True

optimizer = optim.Adam([
    {'params': base_params,         'lr': BASE_LR_JOINT},
    {'params': cbam_params,         'lr': CBAM_LR_JOINT},
    {'params': fusion_core_params,  'lr': FUSION_LR},
    {'params': fusion_blend_params, 'lr': FUSION_BLEND_LR},
], betas=(0.9, 0.999))

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['training']['lr_step'],
    gamma=config['training']['lr_decay']
 )

print("\nOptimizer rebuilt after weight load:")
print(f"  base={BASE_LR_JOINT}  cbam={CBAM_LR_JOINT}  fusion_core={FUSION_LR}  fusion_blend={FUSION_BLEND_LR}")
print(f"  LR decay: {config['training']['lr_decay']} every {config['training']['lr_step']} epochs")

# ============================================================================
# Training loop
# ============================================================================

best_loss = float('inf')
best_val_loss = float('inf')
train_losses, train_mse_losses, train_range_losses = [], [], []
train_tv_losses, train_edge_tv_losses, train_freq_losses = [], [], []
train_dir_losses, train_axis_freq_losses = [], []
train_dir_x_losses, train_dir_y_losses = [], []
val_losses = []
epoch_times = []

overfit_epochs = 0
OVERFIT_PATIENCE = 3
OVERFIT_MIN_DELTA = 0.0

pbar = tqdm(
    range(1, num_epochs + 1),
    desc="Training", unit="epoch", file=sys.stdout, ncols=110,
    bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
)
for epoch in pbar:
    epoch_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # ---- Batch loop ----
    model.train()
    for diag in [cbam_diag, fusion_diag]:
        for key in diag:
            diag[key].clear()

    b_loss, b_mse, b_range, b_tv, b_etv, b_freq = [], [], [], [], [], []
    b_dir, b_axis_freq, b_dir_x, b_dir_y = [], [], [], []

    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (lrs, alphas, hrs, hr_maps, names) in enumerate(train_loader, start=1):
        lrs    = lrs.float().to(device, non_blocking=True)
        alphas = alphas.float().to(device, non_blocking=True)
        hrs    = hrs.float().to(device, non_blocking=True)

        amp_ctx = torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP) if torch.cuda.is_available() else nullcontext()
        with amp_ctx:
            sr_output = model(lrs, alphas)

            if epoch == 1 and batch_idx == 1:
                pbar.write(f"Shape check: LR={tuple(lrs.shape)} SR={tuple(sr_output.shape)} HR={tuple(hrs.shape)}")

            mse_loss   = criterion(sr_output, hrs)
            range_loss = compute_range_loss(sr_output, hrs)

            zero_reg = torch.zeros_like(mse_loss)
            tv_reg = zero_reg
            etv_reg = zero_reg
            freq_reg = zero_reg
            dir_reg = zero_reg
            dir_x_reg = zero_reg
            dir_y_reg = zero_reg
            axis_freq_reg = zero_reg

            if smoothness_mode == 'tv' and lambda_tv > 0:
                tv_reg = tv_loss(sr_output)
            elif smoothness_mode == 'edge_aware_tv' and lambda_edge_tv > 0:
                etv_reg = edge_aware_tv_loss(sr_output, hrs, edge_k=edge_aware_k)
            elif smoothness_mode == 'freq_band' and lambda_freq > 0:
                freq_reg = freq_band_loss(sr_output, hrs, freq_low_ratio, freq_high_ratio, freq_use_log)

            if lambda_dir > 0:
                dir_reg, dir_x_reg, dir_y_reg = directional_grad_loss(
                    sr_output,
                    hrs,
                    emphasis_axis=dir_emphasis_axis,
                    emphasis=dir_emphasis,
                )

            if lambda_axis_freq > 0:
                axis_freq_reg = axis_freq_loss(
                    sr_output,
                    hrs,
                    low_ratio=freq_low_ratio,
                    high_ratio=freq_high_ratio,
                    axis_sigma_ratio=axis_sigma_ratio,
                    use_log_magnitude=freq_use_log,
                )

            smooth_loss = {'edge_aware_tv': etv_reg, 'tv': tv_reg, 'freq_band': freq_reg}.get(
                smoothness_mode, zero_reg)

            loss_total = (
                mse_loss
                + lambda_range * range_loss
                + lambda_smooth * smooth_loss
                + lambda_dir * dir_reg
                + lambda_axis_freq * axis_freq_reg
            )

        loss_for_log = loss_total.detach()
        loss = loss_total / GRAD_ACCUM_STEPS

        if scaler_enabled:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (batch_idx % GRAD_ACCUM_STEPS == 0) or (batch_idx == len(train_loader)):
            if scaler_enabled:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        b_loss.append(float(loss_for_log.item()))
        b_mse.append(float(mse_loss.detach().item()))
        b_range.append(float(range_loss.detach().item()))
        b_tv.append(float(tv_reg.detach().item()))
        b_etv.append(float(etv_reg.detach().item()))
        b_freq.append(float(freq_reg.detach().item()))
        b_dir.append(float(dir_reg.detach().item()))
        b_axis_freq.append(float(axis_freq_reg.detach().item()))
        b_dir_x.append(float(dir_x_reg.detach().item()))
        b_dir_y.append(float(dir_y_reg.detach().item()))

    scheduler.step()

    # ---- Epoch aggregation ----
    epoch_loss      = float(np.mean(b_loss))
    epoch_mse       = float(np.mean(b_mse))
    epoch_range     = float(np.mean(b_range))
    epoch_tv        = float(np.mean(b_tv))
    epoch_etv       = float(np.mean(b_etv))
    epoch_freq      = float(np.mean(b_freq))
    epoch_dir       = float(np.mean(b_dir))
    epoch_axis_freq = float(np.mean(b_axis_freq))
    epoch_dir_x     = float(np.mean(b_dir_x))
    epoch_dir_y     = float(np.mean(b_dir_y))
    epoch_time      = time.time() - epoch_start

    train_losses.append(epoch_loss)
    train_mse_losses.append(epoch_mse)
    train_range_losses.append(epoch_range)
    train_tv_losses.append(epoch_tv)
    train_edge_tv_losses.append(epoch_etv)
    train_freq_losses.append(epoch_freq)
    train_dir_losses.append(epoch_dir)
    train_axis_freq_losses.append(epoch_axis_freq)
    train_dir_x_losses.append(epoch_dir_x)
    train_dir_y_losses.append(epoch_dir_y)
    epoch_times.append(epoch_time)

    # ---- Validation (MSE only) ----
    val_loss = None
    if val_loader is not None:
        model.eval()
        v_mse = []
        with torch.no_grad():
            for lrs, alphas, hrs, hr_maps, names in val_loader:
                lrs    = lrs.float().to(device, non_blocking=True)
                alphas = alphas.float().to(device, non_blocking=True)
                hrs    = hrs.float().to(device, non_blocking=True)

                amp_ctx = torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP) if torch.cuda.is_available() else nullcontext()
                with amp_ctx:
                    sr_output = model(lrs, alphas)
                    mse_loss = criterion(sr_output, hrs)
                v_mse.append(float(mse_loss.detach().item()))
        val_loss = float(np.mean(v_mse)) if v_mse else None
        val_losses.append(val_loss)
        if val_loss is not None and val_loss < best_val_loss:
            best_val_loss = val_loss

    # ---- Overfitting heuristic ----
    if val_loss is not None and len(train_losses) >= 2 and len(val_losses) >= 2:
        train_down = train_losses[-1] < train_losses[-2]
        val_up = val_losses[-1] > (min(val_losses[:-1]) + OVERFIT_MIN_DELTA)
        if train_down and val_up:
            overfit_epochs += 1
        else:
            overfit_epochs = 0
        if overfit_epochs >= OVERFIT_PATIENCE:
            pbar.write("WARNING: possible overfitting (train loss down, val loss up).")

    # ---- Progress bar ----
    postfix = {
        'loss': f'{epoch_loss:.6f}',
        'mse': f'{epoch_mse:.6f}',
        'range': f'{epoch_range:.4f}',
        'freq': f'{epoch_freq:.4f}',
        'dir': f'{epoch_dir:.4f}',
        'time': f'{epoch_time:.1f}s'
    }
    if val_loss is not None:
        postfix['val'] = f'{val_loss:.6f}'
    if cbam_diag["spatial_stds"]:
        postfix['c_sstd'] = f"{np.mean(cbam_diag['spatial_stds']):.4f}"
    if fusion_diag["blend_values"]:
        postfix['fa_blend'] = f"{np.mean(fusion_diag['blend_values']):.4f}"
    if torch.cuda.is_available():
        postfix['vram'] = f"{torch.cuda.max_memory_allocated()/(1024**3):.2f}G"
    pbar.set_postfix(postfix)

    # ---- Periodic diagnostics ----
    if epoch == 1 or epoch % 10 == 0 or epoch == num_epochs:
        lines = [f"=== Epoch {epoch}/{num_epochs} ==="]
        lines.append(f"  Train loss: {epoch_loss:.6f}  MSE: {epoch_mse:.6f}")
        if val_loss is not None:
            lines.append(f"  Val loss (MSE): {val_loss:.6f}")

        if cbam_diag["spatial_means"]:
            ch_m = np.mean(cbam_diag["channel_means"])
            ch_s = np.mean(cbam_diag["channel_stds"])
            sp_m = np.mean(cbam_diag["spatial_means"])
            sp_s = np.mean(cbam_diag["spatial_stds"])
            lines.append(f"  CBAM  channel(mean/std): {ch_m:.4f}/{ch_s:.4f}  spatial(mean/std): {sp_m:.4f}/{sp_s:.4f}")

        if fusion_diag["blend_values"]:
            fa_blend = np.mean(fusion_diag["blend_values"])
            fa_std   = np.std(fusion_diag["blend_values"])
            status   = "learning" if fa_blend > 0.01 else "passthrough"
            lines.append(f"  FusionAttn blend: {fa_blend:.4f} +- {fa_std:.4f}  ({status})")

        lines.append(f"  Directional loss: total={epoch_dir:.5f}  x={epoch_dir_x:.5f}  y={epoch_dir_y:.5f}")
        lines.append(f"  Axis-freq loss: {epoch_axis_freq:.5f}")

        if torch.cuda.is_available():
            lines.append(f"  Peak VRAM this epoch: {torch.cuda.max_memory_allocated()/(1024**3):.2f} GB")

        for line in lines:
            pbar.write(line)

    sys.stdout.flush()

    # ---- Save best + checkpoints ----
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), best_weights_path)
        pbar.write(f"  Best loss updated at epoch {epoch}: {best_loss:.6f}")

    if epoch % 10 == 0:
        ckpt_path = checkpoint_dir / f"checkpoint_epoch_{epoch}.pth"
        torch.save(model.state_dict(), ckpt_path)
        pbar.write(f"  Checkpoint saved: epoch_{epoch}.pth")

        meta = {
            'run_name': run_name,
            'source_weight': str(selected_source_weight_path) if selected_source_weight_path else None,
            'source_has_fusion_keys': bool(source_has_fusion_keys),
            'last_saved_epoch': epoch,
            'best_loss': float(best_loss),
            'best_val_loss': float(best_val_loss) if best_val_loss < float('inf') else None,
            'current_loss': epoch_loss,
            'current_mse': epoch_mse,
            'current_val_loss': val_loss,
            'current_range': epoch_range,
            'current_freq': epoch_freq,
            'current_dir': epoch_dir,
            'current_axis_freq': epoch_axis_freq,
            'smoothness_mode': smoothness_mode,
            'lambda_smooth': float(lambda_smooth),
            'lambda_dir': float(lambda_dir),
            'dir_emphasis_axis': dir_emphasis_axis,
            'dir_emphasis': float(dir_emphasis),
            'lambda_axis_freq': float(lambda_axis_freq),
            'axis_sigma_ratio': float(axis_sigma_ratio),
            'freq_low_ratio': freq_low_ratio,
            'freq_high_ratio': freq_high_ratio,
            'num_epochs': num_epochs,
            'dataset_size': len(train_dataset),
            'use_amp': bool(USE_AMP),
            'amp_dtype': AMP_DTYPE_NAME,
            'grad_accum_steps': int(GRAD_ACCUM_STEPS),
            'timestamp': time.time()
        }
        with open(checkpoint_meta_path, 'w') as f:
            json_module.dump(meta, f, indent=2)

pbar.close()

for hook in all_hooks:
    hook.remove()
all_hooks.clear()

# ============================================================================
# Summary
# ============================================================================

print("\n" + "=" * 70)
print(f"Training complete  |  Best loss: {best_loss:.6f}")
if best_val_loss < float('inf'):
    print(f"  Best val (MSE): {best_val_loss:.6f}")
print(f"  Total: {sum(epoch_times)/60:.1f} min  |  Avg/epoch: {np.mean(epoch_times):.1f}s")
if len(train_mse_losses) >= 2 and train_mse_losses[0] > 0:
    improvement = (train_mse_losses[0] - train_mse_losses[-1]) / train_mse_losses[0] * 100
else:
    improvement = 0.0
print(f"  MSE:   {train_mse_losses[0]:.6f} -> {train_mse_losses[-1]:.6f}  ({improvement:.1f}% improvement)")
print(f"  Range: {train_range_losses[0]:.6f} -> {train_range_losses[-1]:.6f}")
print(f"  Freq:  {train_freq_losses[0]:.6f} -> {train_freq_losses[-1]:.6f}")
print(f"  Dir:   {train_dir_losses[0]:.6f} -> {train_dir_losses[-1]:.6f}  (x={train_dir_x_losses[-1]:.6f}, y={train_dir_y_losses[-1]:.6f})")
print(f"  AxisF: {train_axis_freq_losses[0]:.6f} -> {train_axis_freq_losses[-1]:.6f}")
print(f"  Weights: {best_weights_path}")
print("=" * 70)

last_trained_weight_path = best_weights_path
candidate_summary = {
    "run_name": run_name,
    "run_dir": str(run_dir),
    "best_weight_path": str(best_weights_path),
    "checkpoint_dir": str(checkpoint_dir),
    "meta_path": str(checkpoint_meta_path),
    "source_weight": str(selected_source_weight_path) if selected_source_weight_path else None,
    "arch_family": ACTIVE_ARCH_FAMILY,
    "created_at": datetime.now().isoformat(),
    "best_training_loss": float(best_loss),
    "best_val_loss": float(best_val_loss) if best_val_loss < float('inf') else None
}
with open(LAST_TRAINING_CANDIDATE_PATH, "w") as f:
    json.dump(candidate_summary, f, indent=2)

print("\nDone. Use weights_manager.ipynb to promote or discard this run.")

## 9. Plot Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training/validation loss
axes[0].plot(train_losses, linewidth=2, color='blue', label='train')
if len(val_losses) > 0:
    axes[0].plot(val_losses, linewidth=2, color='green', label='val')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Loss Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')
if len(val_losses) > 0:
    axes[0].legend()

# Epoch time
axes[1].plot(epoch_times, linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Epoch Computation Time', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Statistics:")
print(f"  Total time: {sum(epoch_times)/60:.1f} minutes")
print(f"  Avg time per epoch: {np.mean(epoch_times):.1f}s")
print(f"  Loss decreased by: {(train_losses[0] - train_losses[-1]) / train_losses[0] * 100:.1f}%")
if len(val_losses) > 0:
    print(f"  Best val (MSE): {min(val_losses):.6f}")